In [ ]:
import pandas as pd

roll = 1024170194
#n soma sekhar 
s = str(roll)

digits = []
for ch in s:
    digits.append(int(ch))

print ("Digits: ", digits)

n = len(digits)
d1 = digits[n-2]
d2 = digits[n-1]
print ("Last two digits: ", d1, d2)

categories = ["billing", "account", "general"]

cat1 = categories[d1 % 3]
cat2 = categories[d2 % 3]

print ("digit", d1, "-> category[", d1, "% 3] =", cat1)
print ("digit", d2, "-> category[", d2, "% 3] =", cat2)

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee", "category": "billing"}
]

billing = [
    {"question": "how do i download my invoice",
     "answer": "Go to the Billing section and click Download Invoice.",
     "keywords": "invoice download receipt"},
    {"question": "why was i charged twice",
     "answer": "The extra amount will be refunded within 5-7 working days.",
     "keywords": "refund duplicate charged"}
]

account = [
    {"question": "how do i update my registered mobile number",
     "answer": "Go to Profile > Contact Details and verify the new number with OTP.",
     "keywords": "mobile number update"},
    {"question": "how do i delete my account",
     "answer": "Go to Settings > Account > Delete Account.",
     "keywords": "delete close account"}
]

general = [
    {"question": "where is your office located",
     "answer": "Our office is at MG Road, Bengaluru.",
     "keywords": "office address location"},
    {"question": "how can i contact customer support",
     "answer": "Call 1800-123-456 or email us from the Help page.",
     "keywords": "contact support helpline"}
]

if cat1 == "billing":
    q1 = billing[0]
elif cat1 == "account":
    q1 = account[0]
else:
    q1 = general[0]

if cat2 == "billing":
    q2 = billing[1]
elif cat2 == "account":
    q2 = account[1]
else:
    q2 = general[1]

e1 = {
    "question": q1["question"],
    "answer": q1["answer"],
    "keywords": q1["keywords"],
    "category": cat1
}

e2 = {
    "question": q2["question"],
    "answer": q2["answer"],
    "keywords": q2["keywords"],
    "category": cat2
}

personalized_entries = [e1, e2]

entries = []
for e in fixed_entries:
    entries.append(e)
for e in personalized_entries:
    entries.append(e)

df = pd.DataFrame(entries)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)

print ("Rows: ", len(df))
print (df)

Digits:  [1, 0, 2, 4, 1, 7, 0, 1, 9, 9]
Last two digits:  9 9
digit 9 -> category[ 9 % 3] = billing
digit 9 -> category[ 9 % 3] = billing
Rows:  6
                       question                                                      answer                  keywords category
0        what is the annual fee                                   The annual fee is Rs 500.     fee cost price charge  billing
1         how to reset password                            Go to Settings > Reset Password.      password reset login  account
2   what are your working hours                                   We are open 9 AM to 5 PM.    hours timing open time  general
3         how can i pay the fee                  You can pay via UPI, card, or net banking.       pay payment upi fee  billing
4  how do i download my invoice       Go to the Billing section and click Download Invoice.  invoice download receipt  billing
5       why was i charged twice  The extra amount will be refunded within 5-7 working days.

In [2]:
def score_query(query, df):
    query = query.lower()
    query = query.replace("?", "")
    query = query.replace(",", "")
    query = query.replace(".", "")
    words = query.split()

    uniq = []
    for w in words:
        if w not in uniq:
            uniq.append(w)

    results = []
    total = len(uniq)

    for i in range(len(df)):
        q = df.loc[i, "question"]
        a = df.loc[i, "answer"]
        kw = df.loc[i, "keywords"].lower().split()

        score = 0
        matched = []
        for w in uniq:
            found = False
            for k in kw:
                if w == k:
                    found = True
            if found == True:
                score = score + 1
                matched.append(w)

        if score > 0:
            conf = round(score / total * 100, 2)
            results.append([q, a, score, conf, matched])

    for i in range(len(results)):
        for j in range(len(results) - i - 1):
            if results[j][2] < results[j+1][2]:
                temp = results[j]
                results[j] = results[j+1]
                results[j+1] = temp

    return results


query = input("Enter a query: ")
res = score_query(query, df)

if len(res) == 0:
    print ("No matching entries found")
else:
    rank = 1
    for r in res:
        print ("Rank", rank, "|", r[0], "| score:", r[2], "| confidence:", str(r[3]) + "%", "| matched:", r[4])
        rank = rank + 1

Enter a query:  how can i pay the fee
Rank 1 | how can i pay the fee | score: 2 | confidence: 33.33% | matched: ['pay', 'fee']
Rank 2 | what is the annual fee | score: 1 | confidence: 16.67% | matched: ['fee']


In [3]:
def same_category(category_name, df):
    qs = []
    for i in range(len(df)):
        if df.loc[i, "category"] == category_name:
            qs.append(df.loc[i, "question"])
    return qs


cat = e1["category"]
qs = same_category(cat, df)

print ("Questions in category: ", cat)
print (qs)

Questions in category:  billing
['what is the annual fee', 'how can i pay the fee', 'how do i download my invoice', 'why was i charged twice']


In [4]:
row = 0
print ("Entry: ", df.loc[row, "question"])
print ("Old keywords: ", df.loc[row, "keywords"])

new_kw = input("Enter a new keyword: ")
new_kw = new_kw.strip().lower()

old = df.loc[row, "keywords"]
old_list = old.split()

if new_kw == "":
    print ("Nothing entered")
elif new_kw in old_list:
    print ("Keyword already present")
else:
    df.loc[row, "keywords"] = old + " " + new_kw
    print ("Keyword added")

print ("New keywords: ", df.loc[row, "keywords"])

filename = str(roll) + "_faq_data.csv"
df.to_csv(filename, index=False)

df2 = pd.read_csv(filename)
print ("Saved as: ", filename)
print (df2)

Entry:  what is the annual fee
Old keywords:  fee cost price charge
Enter a new keyword:  money
Keyword added
New keywords:  fee cost price charge money
Saved as:  1024170199_faq_data.csv
                       question                                                      answer                     keywords category
0        what is the annual fee                                   The annual fee is Rs 500.  fee cost price charge money  billing
1         how to reset password                            Go to Settings > Reset Password.         password reset login  account
2   what are your working hours                                   We are open 9 AM to 5 PM.       hours timing open time  general
3         how can i pay the fee                  You can pay via UPI, card, or net banking.          pay payment upi fee  billing
4  how do i download my invoice       Go to the Billing section and click Download Invoice.     invoice download receipt  billing
5       why was i charged twice 

In [5]:
counts = df.groupby("category").size()

print ("Number of FAQ entries per category: ")
print (counts)

total = 0
for c, cnt in counts.items():
    total = total + cnt
print ("Total entries: ", total)

Number of FAQ entries per category: 
category
account    1
billing    4
general    1
dtype: int64
Total entries:  6


In [6]:
def score_query_with_ties(query, df):
    res = score_query(query, df)

    print ("Query: ", query)

    if len(res) == 0:
        print ("No matching entries found")
        print ()
        return

    high = 0
    for r in res:
        if r[2] > high:
            high = r[2]

    top = []
    for r in res:
        if r[2] == high:
            top.append(r)

    print ("Highest Score: ", high)

    if len(top) > 1:
        print ("Tie found between", len(top), "entries:")
        for t in top:
            print (t[0], "->", t[1])
    else:
        print ("Only one best match:")
        print (top[0][0], "->", top[0][1])
    print ()


print ("Tie demonstration: ")
score_query_with_ties("fee", df)

print ("No tie demonstration: ")
score_query_with_ties("reset my password", df)

Tie demonstration: 
Query:  fee
Highest Score:  1
Tie found between 2 entries:
what is the annual fee -> The annual fee is Rs 500.
how can i pay the fee -> You can pay via UPI, card, or net banking.

No tie demonstration: 
Query:  reset my password
Highest Score:  2
Only one best match:
how to reset password -> Go to Settings > Reset Password.

